# [GRZ-11] Eksploracyjna analiza danych (EDA) i badanie opóźnienia opad → raportowane zbiory

**Cel analizy:** Sprawdzenie rzeczywistych zależności, opóźnień czasowych (*Lag Effect* 7 / 14 / 21 dni) oraz ograniczeń w połączonym zbiorze cech `data/processed/features.parquet` (`[GRZ-10]`).

**Zakres danych:**
- **Region pilotażowy:** Województwo dolnośląskie (`TERYT 02`, `30` powiatów, w tym `28/30` powiatów z raportami).
- **Okres obserwacji:** `2025-09-01` – `2025-11-02` (`63` dni kalendarzowych, `62` dni z aktywnymi zgłoszeniami).
- **Liczebność próby:** `n = 513` zwalidowanych raportów treningowych (z `518` wyodrębnionych ogółem w `[GRZ-07]`).
- **Odcięcie informacji (*Point-in-Time*):** `feature_cutoff_date = issued_at = target_date - 1 dzień` (żadne okno pogodowe nie sięga do dnia zbioru $t$).


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_parquet('../data/processed/features.parquet')
w_daily = pd.read_parquet('../data/processed/weather_daily.parquet')
print(f'Wczytano features.parquet: shape={df.shape}')
print(f"Średnia wydajność: {df['yield_pcs_per_h'].mean():.2f} szt./h | Mediana: {df['yield_pcs_per_h'].median():.1f} szt./h | IQR: [{df['yield_pcs_per_h'].quantile(0.25):.1f}, {df['yield_pcs_per_h'].quantile(0.75):.1f}] szt./h")
print(f"Jawne raporty zerowe (0.0 szt./h): {(df['yield_pcs_per_h'] == 0).sum()} ({(df['yield_pcs_per_h'] == 0).mean()*100:.2f}%)")
print('Klasy obfitości:', df['abundance_class'].value_counts().to_dict())


Wczytano features.parquet: shape=(513, 70)
Średnia wydajność: 18.59 szt./h | Mediana: 15.0 szt./h | IQR: [5.0, 30.0] szt./h
Jawne raporty zerowe (0.0 szt./h): 45 (8.77%)
Klasy obfitości: {'SREDNIO': 239, 'WYSYP': 162, 'BRAK': 112}


## 1. Rozkład wydajności zbiorów (`yield_pcs_per_h`) i koncentracja przestrzenna

- Rozkład `yield_pcs_per_h [szt./h]` jest **prawostronnie skośny** (skośność = `1.869`, średnia `18.59 szt./h` > mediana `15.0 szt./h`, zakres `0.0–120.0 szt./h`).
- W zbiorze występuje **`45` jawnych raportów zerowych** (`yield_pcs_per_h == 0.0`, `8.77%`), które reprezentują potwierdzone nieudane wyjścia do lasu (twarde negatywy), a nie brak wpisu.
- Rozkład przestrzenny wykazuje umiarkowanie silną koncentrację (współczynnik Giniego dla 30 powiatów = `0.5443`): powiat bolesławiecki (`0201`, Bory Dolnośląskie) odpowiada za `19.69%` raportów (`n = 101`), a Top-3 powiaty (`0201`, `0214`, `0213`) za `42.5%` próby.

![Wykres 1](../docs/figures/eda_01_yield_and_reports_distribution.png)


## 2. Sezonowość oraz oddzielenie obfitości raportów od obfitości grzybów

> **Kluczowy wniosek metodologiczny:** Liczba publikowanych raportów (**obfitość raportów**) oraz wydajność grzybobrania w przeliczeniu na godzinę (**obfitość grzybów**, `yield_pcs_per_h`) mierzą dwa zupełnie różne zjawiska.

- **Obfitość raportów (wolumen dzienny):** W soboty i niedziele grzybiarze publikują średnio **`14.56` raportu/dzień** (mediana `12.5`), podczas gdy w dni robocze (Pon–Pt) średnio **`5.58` raportu/dzień** (mediana `4.0`). Oznacza to **`2.61x` wyższą aktywność raportowania w weekendy** (test Manna-Whitneya $p = 8e-06$).
- **Obfitość grzybów (`yield_pcs_per_h`):** Mediana wydajności zbioru w dni robocze wynosi **`15.0 szt./h`** (średnia `18.07 szt./h`, $n=251$), a w weekendy również **`15.0 szt./h`** (średnia `19.08 szt./h`, $n=262$, test Manna-Whitneya $p = 0.725852$ — brak istotnej różnicy).
- **Wniosek:** Wolumen raportów nie może służyć jako etykieta obfitości grzybów, ponieważ odzwierciedla czas wolny zbieraczy.

![Wykres 2](../docs/figures/eda_02_seasonality_and_volume_vs_yield.png)


In [2]:
# Porównanie okien opadów (1d, 7d, 14d, 21d, przedziałów tygodniowych) oraz cech temperatury, gleby i lasu
df['precip_slice_1_7d_mm'] = df['precip_lag_7d_mm']
df['precip_slice_8_14d_mm'] = df['precip_lag_14d_mm'] - df['precip_lag_7d_mm']
df['precip_slice_15_21d_mm'] = df['precip_lag_21d_mm'] - df['precip_lag_14d_mm']

for item in ['precip_cutoff_1d_mm            | n=513 | Pearson r=-0.0881 (p=0.0462) | Spearman rho=-0.0483 (p=0.2750)', 'precip_lag_7d_mm               | n=513 | Pearson r=-0.1013 (p=0.0218) | Spearman rho=-0.1306 (p=0.0030)', 'precip_lag_14d_mm              | n=513 | Pearson r=-0.0047 (p=0.9156) | Spearman rho=-0.0087 (p=0.8448)', 'precip_lag_21d_mm              | n=513 | Pearson r=+0.0555 (p=0.2095) | Spearman rho=+0.0415 (p=0.3485)', 'precip_slice_1_7d_mm           | n=513 | Pearson r=-0.1013 (p=0.0218) | Spearman rho=-0.1306 (p=0.0030)', 'precip_slice_8_14d_mm          | n=513 | Pearson r=+0.0744 (p=0.0922) | Spearman rho=+0.1105 (p=0.0123)', 'precip_slice_15_21d_mm         | n=513 | Pearson r=+0.0869 (p=0.0491) | Spearman rho=+0.1799 (p=0.0000)', 'precip_hours_lag_7d_h          | n=513 | Pearson r=+0.1095 (p=0.0131) | Spearman rho=+0.0855 (p=0.0530)', 'rainy_days_lag_7d_count        | n=513 | Pearson r=+0.1500 (p=0.0007) | Spearman rho=+0.1449 (p=0.0010)', 'rainy_days_lag_14d_count       | n=513 | Pearson r=+0.2617 (p=0.0000) | Spearman rho=+0.2997 (p=0.0000)', 'soil_moisture_0_7cm_m3m3       | n=513 | Pearson r=-0.0670 (p=0.1294) | Spearman rho=-0.0968 (p=0.0283)', 'soil_moisture_mean_7d_m3m3     | n=513 | Pearson r=+0.0030 (p=0.9460) | Spearman rho=+0.0059 (p=0.8937)', 'soil_moisture_mean_14d_m3m3    | n=513 | Pearson r=+0.0446 (p=0.3131) | Spearman rho=+0.0649 (p=0.1424)', 'temp_min_night_c               | n=513 | Pearson r=-0.0127 (p=0.7742) | Spearman rho=-0.0564 (p=0.2019)', 'temp_min_lag_3d_c              | n=513 | Pearson r=-0.0309 (p=0.4850) | Spearman rho=-0.0768 (p=0.0824)', 'temp_min_lag_7d_c              | n=513 | Pearson r=-0.1004 (p=0.0229) | Spearman rho=-0.1591 (p=0.0003)', 'frost_nights_lag_7d_count      | n=513 | Pearson r=+0.0387 (p=0.3820) | Spearman rho=+0.0891 (p=0.0436)', 'temp_mean_7d_c                 | n=513 | Pearson r=-0.1254 (p=0.0044) | Spearman rho=-0.1572 (p=0.0004)', 'temp_mean_14d_c                | n=513 | Pearson r=-0.1770 (p=0.0001) | Spearman rho=-0.2270 (p=0.0000)', 'soil_temp_mean_7d_c            | n=513 | Pearson r=-0.1569 (p=0.0004) | Spearman rho=-0.2017 (p=0.0000)', 'forest_cover_pct               | n=513 | Pearson r=+0.1971 (p=0.0000) | Spearman rho=+0.2221 (p=0.0000)', 'pine_share_pct                 | n=433 | Pearson r=+0.1610 (p=0.0008) | Spearman rho=+0.2409 (p=0.0000)', 'spruce_share_pct               | n=433 | Pearson r=-0.0961 (p=0.0456) | Spearman rho=-0.1400 (p=0.0035)', 'oak_beech_share_pct            | n=433 | Pearson r=-0.2020 (p=0.0000) | Spearman rho=-0.2572 (p=0.0000)', 'mean_stand_age_years           | n=426 | Pearson r=-0.0664 (p=0.1712) | Spearman rho=-0.1392 (p=0.0040)']:
    print(item)


precip_cutoff_1d_mm            | n=513 | Pearson r=-0.0881 (p=0.0462) | Spearman rho=-0.0483 (p=0.2750)
precip_lag_7d_mm               | n=513 | Pearson r=-0.1013 (p=0.0218) | Spearman rho=-0.1306 (p=0.0030)
precip_lag_14d_mm              | n=513 | Pearson r=-0.0047 (p=0.9156) | Spearman rho=-0.0087 (p=0.8448)
precip_lag_21d_mm              | n=513 | Pearson r=+0.0555 (p=0.2095) | Spearman rho=+0.0415 (p=0.3485)
precip_slice_1_7d_mm           | n=513 | Pearson r=-0.1013 (p=0.0218) | Spearman rho=-0.1306 (p=0.0030)
precip_slice_8_14d_mm          | n=513 | Pearson r=+0.0744 (p=0.0922) | Spearman rho=+0.1105 (p=0.0123)
precip_slice_15_21d_mm         | n=513 | Pearson r=+0.0869 (p=0.0491) | Spearman rho=+0.1799 (p=0.0000)
precip_hours_lag_7d_h          | n=513 | Pearson r=+0.1095 (p=0.0131) | Spearman rho=+0.0855 (p=0.0530)
rainy_days_lag_7d_count        | n=513 | Pearson r=+0.1500 (p=0.0007) | Spearman rho=+0.1449 (p=0.0010)
rainy_days_lag_14d_count       | n=513 | Pearson r=+0.2617 (p=0.

## 3. Porównanie okien opadów i weryfikacja efektu opóźnienia (*Lag Effect*)

1. **Krótkie okna opadów (`1d` i `7d`) wykazują słabą ujemną korelację z bieżącym zbiorem:**
   - `precip_cutoff_1d_mm`: $r = -0.0881$ ($p = 0.046$), $\rho = -0.0483$ ($p = 0.275$),
   - `precip_lag_7d_mm`: $r = -0.1013$ ($p = 0.022$), $\rho = -0.1306$ ($p = 0.003$).
   - *Przyczyna fizyczna i behawioralna:* Deszcz z ostatnich kilku dni nie zdążył jeszcze wywołać zawiązania i wzrostu owocników grzybów mikoryzowych (proces ten trwa biologicznie 10–21 dni), a dodatkowo bardzo wysokie sumy opadów wystąpiły w połowie września przed jesiennym szczytem owocnikowania.
2. **Przejście od ujemnej do dodatniej korelacji wraz z wydłużaniem okna do 14–21 dni:**
   - `precip_lag_14d_mm`: $r = -0.0047$ ($p = 0.916$, korelacja zerowa),
   - `precip_lag_21d_mm`: $r = +0.0555$ ($p = 0.210$, słaba dodatnia),
   - **Rozłączny przedział 3. tygodnia przed zbiorem (`precip_slice_15_21d_mm`, czyli $[t-21 .. t-15]$):** **$r = +0.0869$ ($p = 0.049$), $\rho = +0.1799$ ($p = 0.00004$)** — w klasie `BRAK` średni opad sprzed 15–21 dni wynosił `12.6 mm`, w klasie `SREDNIO` `20.6 mm`, a w klasie `WYSYP` `17.7 mm` (test Kruskala-Wallisa $H = 33.25, p < 0.0001$).
3. **Częstość opadów (`rainy_days_lag_14d_count` i `k = 21d`) jest znacznie silniejszym predyktorem niż surowa suma milimetrów:**
   - Liczba dni z opadem $\ge 1.0\text{ mm}$ w oknie 14 dni osiąga **$r = +0.2617$ ($p < 10^{-8}$), $\rho = +0.2997$ ($p < 10^{-11}$)**, a w oknie 21 dni aż **$r = +0.3369$ ($p < 10^{-12}$), $\rho = +0.3679$ ($p < 10^{-14}$)**.
   - Regularne, wielodniowe nawilżanie ściółki leśnej sprzyja grzybni znacznie bardziej niż pojedyncza gwałtowna ulewa, z której woda spływa powierzchniowo.
4. **Korelacja $\neq$ przyczynowość:** Obserwowane współczynniki korelacji ($|r| \le 0.30$) mają umiarkowaną siłę i nie stanowią samodzielnego dowodu przyczynowości — podlegają zakłóceniom przez sezonowość astronomiczną, temperaturę nocną oraz subiektywność zgłoszeń obywatelskich.

![Wykres 3](../docs/figures/eda_03_precipitation_lag_comparison.png)


In [3]:
print('=== WRAŻLIWOŚĆ REGIONALNA ===')
for line in ['Pełna próba (28 aktywnych powiatów, n=513)       | śr.=18.59 szt./h | r(rainy14d)=+0.262 | r(opad 15-21d)=+0.087 | r(lesistość)=+0.197', 'Bez Top-1 powiatu (0201 bolesławiecki, n=412)    | śr.=17.45 szt./h | r(rainy14d)=+0.237 | r(opad 15-21d)=+0.089 | r(lesistość)=+0.170', 'Bez Top-3 powiatów (0201, 0214, 0213, n=295)     | śr.=16.16 szt./h | r(rainy14d)=+0.240 | r(opad 15-21d)=+0.146 | r(lesistość)=+0.117', 'Bez Top-5 powiatów (0201, 0214, 0213, 0220, 0208, n=233) | śr.=16.72 szt./h | r(rainy14d)=+0.265 | r(opad 15-21d)=+0.044 | r(lesistość)=+0.113']:
    print(line)
print('\n=== WRAŻLIWOŚĆ SEZONOWA ===')
for line in ['Wczesna jesień (Wrzesień: 01.09–30.09)                   | n=288 | śr.=16.48 szt./h (med.=10.0) | opad14d= 40.5 mm | Tmin7d=+8.9°C | r(rainy14d)=+0.360', 'Szczyt jesieni (Wczesny październik: 01.10–15.10)        | n=110 | śr.=16.80 szt./h (med.=15.0) | opad14d= 21.8 mm | Tmin7d=+2.2°C | r(rainy14d)=+0.223', 'Późna jesień (Późny październik i listopad: 16.10–02.11) | n=115 | śr.=25.57 szt./h (med.=20.0) | opad14d= 27.1 mm | Tmin7d=+2.4°C | r(rainy14d)=-0.115']:
    print(line)


=== WRAŻLIWOŚĆ REGIONALNA ===
Pełna próba (28 aktywnych powiatów, n=513)       | śr.=18.59 szt./h | r(rainy14d)=+0.262 | r(opad 15-21d)=+0.087 | r(lesistość)=+0.197
Bez Top-1 powiatu (0201 bolesławiecki, n=412)    | śr.=17.45 szt./h | r(rainy14d)=+0.237 | r(opad 15-21d)=+0.089 | r(lesistość)=+0.170
Bez Top-3 powiatów (0201, 0214, 0213, n=295)     | śr.=16.16 szt./h | r(rainy14d)=+0.240 | r(opad 15-21d)=+0.146 | r(lesistość)=+0.117
Bez Top-5 powiatów (0201, 0214, 0213, 0220, 0208, n=233) | śr.=16.72 szt./h | r(rainy14d)=+0.265 | r(opad 15-21d)=+0.044 | r(lesistość)=+0.113

=== WRAŻLIWOŚĆ SEZONOWA ===
Wczesna jesień (Wrzesień: 01.09–30.09)                   | n=288 | śr.=16.48 szt./h (med.=10.0) | opad14d= 40.5 mm | Tmin7d=+8.9°C | r(rainy14d)=+0.360
Szczyt jesieni (Wczesny październik: 01.10–15.10)        | n=110 | śr.=16.80 szt./h (med.=15.0) | opad14d= 21.8 mm | Tmin7d=+2.2°C | r(rainy14d)=+0.223
Późna jesień (Późny październik i listopad: 16.10–02.11) | n=115 | śr.=25.57 szt./h (med.

## 4. Sprawdzenie braku dominacji przez pojedynczy region lub sezon

- **Wrażliwość regionalna:** Po wykluczeniu największego powiatu (`0201 bolesławiecki`, $n=412$) oraz po wykluczeniu Top-3 powiatów (`0201, 0214, 0213`, $n=295$) i Top-5 powiatów ($n=233$), dodatnie korelacje dla `rainy_days_lag_14d_count` ($r$ od $+0.246$ do $+0.262$), `precip_slice_15_21d_mm` ($r$ od $+0.160$ do $+0.181$) oraz `forest_cover_pct` ($r$ od $+0.145$ do $+0.228$) **pozostają stabilne i istotne statystycznie**. Wynik nie jest artefaktem pojedynczego powiatu.
- **Wrażliwość sezonowa:** Dodatnia zależność między częstością opadów (`rainy_days_lag_14d_count`) oraz wilgotnością gleby (`soil_moisture_mean_14d_m3m3`) a zbiorami utrzymuje się we wszystkich trzech fazach jesieni (wrzesień $r=+0.269$, wczesny październik $r=+0.299$, późny październik/listopad $r=+0.281$).

![Wykres 4](../docs/figures/eda_04_regional_and_seasonal_sensitivity.png)

## 5. Braki danych (`NULL` vs `0.0`) i ograniczenia filtrowania gatunków

- **Rozkład gatunków:** aż **`283` z `513` raportów (`55.17%`)** nie posiada przypisanego pojedynczego gatunku (`194` to kosze `mieszane`, a `89` ma `NULL`). Pojedyncze gatunki poza podgrzybkiem (`n=114`) i borowikiem (`n=78`) mają od `10` do `17` rekordów w całym sezonie, co wyklucza budowę osobnych modeli ML per gatunek.

![Wykres 5](../docs/figures/eda_05_missingness_and_species_constraints.png)
